In [ ]:
# Pmod I2S2 Effect Control -- one-cell ipywidgets UI for the Pmod I2S2
# mode-2 path (Pmod Line In -> Pmod ADC -> AudioLab DSP chain ->
# Pmod DAC -> Pmod Line Out, DECISIONS.md D49). Loads AudioLabOverlay
# once, finds the Pmod I2S2 status MMIO from `ip_dict`, forces
# cfg_mode = 2 (DSP) at startup, and exposes the existing effect API
# (set_guitar_effects, set_compressor_settings, set_noise_suppressor_settings,
# set_overdrive_model, OVERDRIVE_MODEL_LABELS, AMP_MODELS,
# DISTORTION_PEDALS_IMPLEMENTED) behind toggles / sliders / dropdowns.
#
# WIRING / SAFETY (READ FIRST):
#   - Pmod I2S2 plugged into PMOD JB (PMOD JB is Pmod-I2S2-only, D48).
#   - Disconnect the on-module Line Out <-> Line In 3.5 mm jumper for
#     mode 2; the DSP loop can feed back if both ends are tied.
#   - Put a real audio source on Pmod Line In at LOW level.
#   - Listen on Pmod Line Out via a separate audio interface, NOT
#     plugged back into Line In.
#   - Mode 0 = tone (default-safe). Mode 1 = ADC->DAC direct loopback
#     (UI requires explicit confirm). Mode 2 = ADC->DSP->DAC (this
#     notebook's default). Mode 3 = mute.
import sys
import time
import traceback

PROJECT_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import ipywidgets as widgets
from IPython.display import display, clear_output
from pynq import MMIO

from audio_lab_pynq.AudioLabOverlay import AudioLabOverlay
from audio_lab_pynq.effect_defaults import (
    DISTORTION_PEDALS_IMPLEMENTED,
    OVERDRIVE_MODELS,
    OVERDRIVE_MODEL_LABELS,
    AMP_MODELS,
)

# ---- Pmod I2S2 status register map (axi_pmod_i2s2_status.v) ------------
REG = {
    "VERSION":      0x00,
    "STATUS":       0x04,
    "FRAME":        0x08,
    "NONZERO":      0x0C,
    "SDOUT_XCOUNT": 0x10,
    "CLIP":         0x14,
    "LAST_LEFT":    0x18,
    "LAST_RIGHT":   0x1C,
    "PEAK_L":       0x20,
    "PEAK_R":       0x24,
    "MODE":         0x28,
    "CLEAR":        0x2C,
}
EXPECTED_VERSION = 0x00480001

MODE = {
    "tone":     0,
    "loopback": 1,
    "dsp":      2,
    "mute":     3,
}
MODE_LABEL = {v: k for k, v in MODE.items()}

# ---- Helpers -------------------------------------------------------------
def _read(mmio, off):
    return mmio.read(off) & 0xFFFFFFFF

def _sign24(x):
    x = x & 0xFFFFFFFF
    if x & 0x80000000:
        return x - (1 << 32)
    return x

def _dbfs(v):
    v = int(v)
    if v <= 0:
        return float("-inf")
    import math
    return 20.0 * math.log10(v / float(1 << 23))

def _find_pmod_status(overlay):
    # PYNQ wraps pmod_status_0 as a DefaultHierarchy whose .read() does not
    # forward to the s_axi MMIO; build the MMIO from ip_dict instead.
    ip_dict = getattr(overlay, "ip_dict", {})
    for key in sorted(ip_dict):
        if "pmod_status" in key or "pmod_i2s2_status" in key:
            entry = ip_dict[key]
            addr = entry.get("phys_addr")
            if addr is None:
                continue
            rng = entry.get("addr_range", 0x10000)
            return MMIO(addr, rng), key
    return None, None

# ---- Overlay + Pmod status ----------------------------------------------
status_out = widgets.Output(layout=widgets.Layout(border="1px solid #444",
                                                  padding="6px"))
log_out = widgets.Output(layout=widgets.Layout(border="1px solid #888",
                                               padding="6px",
                                               max_height="200px",
                                               overflow="auto"))

def _log(msg):
    with log_out:
        ts = time.strftime("%H:%M:%S")
        print("[" + ts + "] " + str(msg))

def _log_exc(msg):
    with log_out:
        ts = time.strftime("%H:%M:%S")
        print("[" + ts + "] ERROR " + str(msg))
        traceback.print_exc()

ovl = None
pmod_mmio = None
pmod_key = None

def load_overlay():
    global ovl, pmod_mmio, pmod_key
    try:
        ovl = AudioLabOverlay()
        _log("AudioLabOverlay loaded.")
    except Exception as e:
        _log_exc("AudioLabOverlay load failed: " + str(e))
        return
    pmod_mmio, pmod_key = _find_pmod_status(ovl)
    if pmod_mmio is None:
        _log("WARN: pmod_status IP not found in ip_dict. "
             "Is this the D49 bit? Mode control disabled.")
    else:
        ver = _read(pmod_mmio, REG["VERSION"])
        if ver != EXPECTED_VERSION:
            _log("WARN: pmod_status VERSION 0x%08X != expected 0x%08X"
                 % (ver, EXPECTED_VERSION))
        else:
            _log("pmod_status at " + str(pmod_key) +
                 ", VERSION 0x%08X." % ver)

def _require_overlay():
    if ovl is None:
        _log("Overlay not loaded -- click 'Load/Reload overlay' first.")
        return False
    return True

def _require_pmod():
    if pmod_mmio is None:
        _log("pmod_status MMIO unavailable.")
        return False
    return True

def write_mode(mode_int, label=None):
    if not _require_pmod():
        return
    try:
        pmod_mmio.write(REG["MODE"], int(mode_int) & 0x3)
        rb = _read(pmod_mmio, REG["MODE"]) & 0x3
        _log("MODE write %d (%s); readback = %d"
             % (mode_int, label or MODE_LABEL.get(mode_int, "?"), rb))
    except Exception as e:
        _log_exc("MODE write failed: " + str(e))

def clear_counters():
    if not _require_pmod():
        return
    try:
        pmod_mmio.write(REG["CLEAR"], 1)
        _log("CLEAR pulse issued.")
    except Exception as e:
        _log_exc("CLEAR failed: " + str(e))

def refresh_status():
    with status_out:
        clear_output(wait=True)
        if pmod_mmio is None:
            print("pmod_status MMIO unavailable.")
            return
        try:
            ver   = _read(pmod_mmio, REG["VERSION"])
            st    = _read(pmod_mmio, REG["STATUS"])
            mode  = (st >> 8) & 0x3
            frame = _read(pmod_mmio, REG["FRAME"])
            nz    = _read(pmod_mmio, REG["NONZERO"])
            xcnt  = _read(pmod_mmio, REG["SDOUT_XCOUNT"])
            clp   = _read(pmod_mmio, REG["CLIP"])
            ll    = _sign24(_read(pmod_mmio, REG["LAST_LEFT"]))
            lr    = _sign24(_read(pmod_mmio, REG["LAST_RIGHT"]))
            pl    = _read(pmod_mmio, REG["PEAK_L"])
            pr    = _read(pmod_mmio, REG["PEAK_R"])
            mode_reg = _read(pmod_mmio, REG["MODE"]) & 0x3
            print("pmod_status IP   : %s" % (pmod_key or "?"))
            print("VERSION          : 0x%08X (expected 0x%08X)" % (ver, EXPECTED_VERSION))
            print("STATUS           : 0x%08X  (mode=%d, sdout_alive=%d, bclk_seen=%d, lrclk_seen=%d)"
                  % (st, mode, (st >> 2) & 1, (st >> 1) & 1, st & 1))
            print("MODE register   : %d (%s)" % (mode_reg, MODE_LABEL.get(mode_reg, "?")))
            print("FRAME_COUNT      : %u" % frame)
            print("NONZERO_COUNT    : %u" % nz)
            print("SDOUT_XCOUNT     : %u" % xcnt)
            print("CLIP_COUNT       : %u" % clp)
            print("LAST_LEFT        : %d" % ll)
            print("LAST_RIGHT       : %d" % lr)
            print("PEAK_ABS_LEFT    : %u  (%.1f dBFS)" % (pl, _dbfs(pl)))
            print("PEAK_ABS_RIGHT   : %u  (%.1f dBFS)" % (pr, _dbfs(pr)))
            print("")
            print("Connection reminder:")
            print("  - Mode 2: DISCONNECT the on-module Line Out <-> Line In jumper.")
            print("  - Put audio source on Line In at LOW volume.")
            print("  - Listen on Line Out via a SEPARATE audio interface.")
        except Exception as e:
            print("refresh_status failed: %s" % e)
            traceback.print_exc()

# ---- Effect state cache --------------------------------------------------
state = {
    # Noise Suppressor
    "noise_gate_on":         False,
    "noise_gate_threshold":  35,
    # Compressor (handled via set_compressor_settings)
    "compressor_enabled":    False,
    "compressor_threshold":  60,
    "compressor_ratio":      30,
    "compressor_response":   40,
    "compressor_makeup":     50,
    # Overdrive
    "overdrive_on":          False,
    "overdrive_model":       0,
    "overdrive_drive":       30,
    "overdrive_tone":        65,
    "overdrive_level":       80,
    # Distortion (pedal-mask based, D6 / D9)
    "distortion_on":         False,
    "distortion_pedal":      "clean_boost",
    "distortion":            25,
    "distortion_tone":       65,
    "distortion_level":      80,
    # Amp
    "amp_on":                False,
    "amp_model":             "clean_combo",
    "amp_input_gain":        35,
    "amp_bass":              50,
    "amp_middle":            50,
    "amp_treble":            50,
    "amp_master":            70,
    # D53: binary drive mode; 0 = current voicing byte, 1 =
    # in-band higher-drive byte for the same amp model.
    "amp_drive_mode":        0,
    # Cab
    "cab_on":                False,
    "cab_model":             1,
    "cab_mix":               100,
    "cab_level":             80,
    # EQ
    "eq_on":                 False,
    "eq_low":                100,
    "eq_mid":                100,
    "eq_high":               100,
    # Reverb
    "reverb_on":             False,
    "reverb_decay":          30,
    "reverb_tone":           65,
    "reverb_mix":            20,
}

def _pedal_mask_for(name):
    """Return a single-pedal mask. `name` must be in DISTORTION_PEDALS_IMPLEMENTED."""
    bit_for = {p: i for i, p in enumerate(DISTORTION_PEDALS_IMPLEMENTED)}
    if name not in bit_for:
        return 0
    return 1 << bit_for[name]

def apply_effects():
    if not _require_overlay():
        return
    try:
        # Compressor + Noise Suppressor have dedicated GPIO setters; the
        # rest of the chain goes through set_guitar_effects in one call.
        ovl.set_compressor_settings(
            threshold=state["compressor_threshold"],
            ratio=state["compressor_ratio"],
            response=state["compressor_response"],
            makeup=state["compressor_makeup"],
            enabled=state["compressor_enabled"],
        )

        # Build the pedal mask only when Distortion is enabled. Mask = 0
        # while disabled keeps the section silent at the byte level.
        pmask = (_pedal_mask_for(state["distortion_pedal"])
                 if state["distortion_on"] else 0)
        # D53: amp_character is no longer forwarded; instead
        # send amp_model_idx + amp_drive_mode and let the
        # overlay derive the character byte.
        amp_model_names = list(AMP_MODELS.keys())
        try:
            amp_model_idx = amp_model_names.index(state["amp_model"])
        except ValueError:
            amp_model_idx = 0
        amp_drive_mode = 1 if int(state.get("amp_drive_mode", 0)) >= 1 else 0

        ovl.set_guitar_effects(
            noise_gate_on=state["noise_gate_on"],
            noise_gate_threshold=state["noise_gate_threshold"],
            overdrive_on=state["overdrive_on"],
            overdrive_model=state["overdrive_model"],
            overdrive_drive=state["overdrive_drive"],
            overdrive_tone=state["overdrive_tone"],
            overdrive_level=state["overdrive_level"],
            distortion_on=state["distortion_on"],
            distortion=state["distortion"],
            distortion_tone=state["distortion_tone"],
            distortion_level=state["distortion_level"],
            distortion_pedal_mask=pmask,
            amp_on=state["amp_on"],
            amp_input_gain=state["amp_input_gain"],
            amp_bass=state["amp_bass"],
            amp_middle=state["amp_middle"],
            amp_treble=state["amp_treble"],
            amp_master=state["amp_master"],
            amp_model_idx=amp_model_idx,
            amp_drive_mode=amp_drive_mode,
            cab_on=state["cab_on"],
            cab_model=state["cab_model"],
            cab_mix=state["cab_mix"],
            cab_level=state["cab_level"],
            eq_on=state["eq_on"],
            eq_low=state["eq_low"],
            eq_mid=state["eq_mid"],
            eq_high=state["eq_high"],
            reverb_on=state["reverb_on"],
            reverb_decay=state["reverb_decay"],
            reverb_tone=state["reverb_tone"],
            reverb_mix=state["reverb_mix"],
        )
        _log("apply_effects OK.")
    except Exception as e:
        _log_exc("apply_effects failed: " + str(e))

def all_effects_off(also_clear_levels=False):
    """Disable every effect ON flag in `state` and re-apply."""
    for k in ("noise_gate_on", "compressor_enabled", "overdrive_on",
             "distortion_on", "amp_on", "cab_on", "eq_on", "reverb_on"):
        state[k] = False
    state["amp_drive_mode"] = 0
    if also_clear_levels:
        state["amp_master"] = 40
        state["cab_level"] = 40
        state["overdrive_level"] = 40
        state["distortion_level"] = 40
    apply_effects()
    _sync_widgets_from_state()

def safe_clean():
    """Mode 2 + all effects off + conservative levels."""
    write_mode(MODE["dsp"], "dsp")
    all_effects_off(also_clear_levels=True)
    refresh_status()
    _log("safe_clean applied: mode 2, all effects off, conservative levels.")

def panic_mute():
    """Mode 3 mute + all effects off."""
    write_mode(MODE["mute"], "mute")
    all_effects_off()
    refresh_status()
    _log("panic_mute applied: mode 3 (mute), all effects off.")

# ---- Widget builders -----------------------------------------------------
SLIDER_W = widgets.Layout(width="260px")

def _bind_toggle(w, key):
    def _cb(change):
        state[key] = bool(change["new"])
    w.observe(_cb, names="value")

def _bind_int_slider(w, key):
    def _cb(change):
        state[key] = int(change["new"])
    w.observe(_cb, names="value")

def _bind_dropdown(w, key):
    def _cb(change):
        state[key] = change["new"]
    w.observe(_cb, names="value")

# Noise Suppressor row
ns_on    = widgets.Checkbox(value=state["noise_gate_on"],
                            description="Noise Sup")
ns_thr   = widgets.IntSlider(value=state["noise_gate_threshold"], min=0, max=100,
                             step=1, description="threshold", layout=SLIDER_W)
_bind_toggle(ns_on, "noise_gate_on")
_bind_int_slider(ns_thr, "noise_gate_threshold")

# Compressor row
comp_on    = widgets.Checkbox(value=state["compressor_enabled"],
                              description="Comp")
comp_thr   = widgets.IntSlider(value=state["compressor_threshold"], min=0, max=100,
                               step=1, description="threshold", layout=SLIDER_W)
comp_ratio = widgets.IntSlider(value=state["compressor_ratio"], min=0, max=100,
                               step=1, description="ratio", layout=SLIDER_W)
comp_resp  = widgets.IntSlider(value=state["compressor_response"], min=0, max=100,
                               step=1, description="response", layout=SLIDER_W)
comp_mk    = widgets.IntSlider(value=state["compressor_makeup"], min=0, max=100,
                               step=1, description="makeup", layout=SLIDER_W)
_bind_toggle(comp_on, "compressor_enabled")
_bind_int_slider(comp_thr, "compressor_threshold")
_bind_int_slider(comp_ratio, "compressor_ratio")
_bind_int_slider(comp_resp, "compressor_response")
_bind_int_slider(comp_mk,   "compressor_makeup")

# Overdrive row
od_on    = widgets.Checkbox(value=state["overdrive_on"], description="Overdrive")
od_model = widgets.Dropdown(
    options=[(lbl, i) for i, lbl in enumerate(OVERDRIVE_MODEL_LABELS)],
    value=state["overdrive_model"], description="model",
)
od_drive = widgets.IntSlider(value=state["overdrive_drive"], min=0, max=100,
                             step=1, description="drive", layout=SLIDER_W)
od_tone  = widgets.IntSlider(value=state["overdrive_tone"], min=0, max=100,
                             step=1, description="tone", layout=SLIDER_W)
od_level = widgets.IntSlider(value=state["overdrive_level"], min=0, max=100,
                             step=1, description="level", layout=SLIDER_W)
_bind_toggle(od_on, "overdrive_on")
_bind_dropdown(od_model, "overdrive_model")
_bind_int_slider(od_drive, "overdrive_drive")
_bind_int_slider(od_tone,  "overdrive_tone")
_bind_int_slider(od_level, "overdrive_level")

# Distortion row
dist_on    = widgets.Checkbox(value=state["distortion_on"], description="Distortion")
dist_pedal = widgets.Dropdown(
    options=list(DISTORTION_PEDALS_IMPLEMENTED),
    value=state["distortion_pedal"], description="pedal",
)
dist_drive = widgets.IntSlider(value=state["distortion"], min=0, max=100,
                               step=1, description="drive", layout=SLIDER_W)
dist_tone  = widgets.IntSlider(value=state["distortion_tone"], min=0, max=100,
                               step=1, description="tone", layout=SLIDER_W)
dist_level = widgets.IntSlider(value=state["distortion_level"], min=0, max=100,
                               step=1, description="level", layout=SLIDER_W)
_bind_toggle(dist_on, "distortion_on")
_bind_dropdown(dist_pedal, "distortion_pedal")
def _dist_drive_cb(change):
    state["distortion"] = int(change["new"])
dist_drive.observe(_dist_drive_cb, names="value")
_bind_int_slider(dist_tone,  "distortion_tone")
_bind_int_slider(dist_level, "distortion_level")

# Amp row
amp_on    = widgets.Checkbox(value=state["amp_on"], description="Amp Sim")
amp_model = widgets.Dropdown(
    options=list(AMP_MODELS.keys()),
    value=state["amp_model"], description="model",
)
amp_in    = widgets.IntSlider(value=state["amp_input_gain"], min=0, max=100,
                              step=1, description="in_gain", layout=SLIDER_W)
amp_bass  = widgets.IntSlider(value=state["amp_bass"], min=0, max=100,
                              step=1, description="bass", layout=SLIDER_W)
amp_mid   = widgets.IntSlider(value=state["amp_middle"], min=0, max=100,
                              step=1, description="mid", layout=SLIDER_W)
amp_treb  = widgets.IntSlider(value=state["amp_treble"], min=0, max=100,
                              step=1, description="treble", layout=SLIDER_W)
amp_mst   = widgets.IntSlider(value=state["amp_master"], min=0, max=100,
                              step=1, description="master", layout=SLIDER_W)
# D53: binary drive mode -- 0 = current voicing, 1 = higher-drive.
amp_drv = widgets.Dropdown(options=[("0", 0), ("1", 1)],
                            value=int(state.get("amp_drive_mode", 0)),
                            description="drv mode")
_bind_toggle(amp_on, "amp_on")
_bind_dropdown(amp_model, "amp_model")
_bind_int_slider(amp_in,   "amp_input_gain")
_bind_int_slider(amp_bass, "amp_bass")
_bind_int_slider(amp_mid,  "amp_middle")
_bind_int_slider(amp_treb, "amp_treble")
_bind_int_slider(amp_mst,  "amp_master")
_bind_dropdown(amp_drv,    "amp_drive_mode")

# Cab row
cab_on    = widgets.Checkbox(value=state["cab_on"], description="Cab IR")
cab_model = widgets.Dropdown(options=[("model 0", 0), ("model 1", 1), ("model 2", 2)],
                             value=state["cab_model"], description="model")
cab_mix   = widgets.IntSlider(value=state["cab_mix"], min=0, max=100,
                              step=1, description="mix", layout=SLIDER_W)
cab_level = widgets.IntSlider(value=state["cab_level"], min=0, max=100,
                              step=1, description="level", layout=SLIDER_W)
_bind_toggle(cab_on, "cab_on")
_bind_dropdown(cab_model, "cab_model")
_bind_int_slider(cab_mix,   "cab_mix")
_bind_int_slider(cab_level, "cab_level")

# EQ row
eq_on    = widgets.Checkbox(value=state["eq_on"], description="EQ")
eq_low   = widgets.IntSlider(value=state["eq_low"], min=0, max=200,
                             step=1, description="low", layout=SLIDER_W)
eq_mid   = widgets.IntSlider(value=state["eq_mid"], min=0, max=200,
                             step=1, description="mid", layout=SLIDER_W)
eq_high  = widgets.IntSlider(value=state["eq_high"], min=0, max=200,
                             step=1, description="high", layout=SLIDER_W)
_bind_toggle(eq_on, "eq_on")
_bind_int_slider(eq_low,  "eq_low")
_bind_int_slider(eq_mid,  "eq_mid")
_bind_int_slider(eq_high, "eq_high")

# Reverb row
rv_on    = widgets.Checkbox(value=state["reverb_on"], description="Reverb")
rv_decay = widgets.IntSlider(value=state["reverb_decay"], min=0, max=100,
                             step=1, description="decay", layout=SLIDER_W)
rv_tone  = widgets.IntSlider(value=state["reverb_tone"], min=0, max=100,
                             step=1, description="tone", layout=SLIDER_W)
rv_mix   = widgets.IntSlider(value=state["reverb_mix"], min=0, max=100,
                             step=1, description="mix", layout=SLIDER_W)
_bind_toggle(rv_on, "reverb_on")
_bind_int_slider(rv_decay, "reverb_decay")
_bind_int_slider(rv_tone,  "reverb_tone")
_bind_int_slider(rv_mix,   "reverb_mix")

ALL_WIDGETS = {
    "noise_gate_on":        ns_on,
    "noise_gate_threshold": ns_thr,
    "compressor_enabled":   comp_on,
    "compressor_threshold": comp_thr,
    "compressor_ratio":     comp_ratio,
    "compressor_response":  comp_resp,
    "compressor_makeup":    comp_mk,
    "overdrive_on":         od_on,
    "overdrive_model":      od_model,
    "overdrive_drive":      od_drive,
    "overdrive_tone":       od_tone,
    "overdrive_level":      od_level,
    "distortion_on":        dist_on,
    "distortion_pedal":     dist_pedal,
    "distortion":           dist_drive,
    "distortion_tone":      dist_tone,
    "distortion_level":     dist_level,
    "amp_on":               amp_on,
    "amp_model":            amp_model,
    "amp_input_gain":       amp_in,
    "amp_bass":             amp_bass,
    "amp_middle":           amp_mid,
    "amp_treble":           amp_treb,
    "amp_master":           amp_mst,
    "amp_drive_mode":       amp_drv,
    "cab_on":               cab_on,
    "cab_model":            cab_model,
    "cab_mix":              cab_mix,
    "cab_level":            cab_level,
    "eq_on":                eq_on,
    "eq_low":               eq_low,
    "eq_mid":               eq_mid,
    "eq_high":              eq_high,
    "reverb_on":            rv_on,
    "reverb_decay":         rv_decay,
    "reverb_tone":          rv_tone,
    "reverb_mix":           rv_mix,
}

def _sync_widgets_from_state():
    for k, w in ALL_WIDGETS.items():
        try:
            w.value = state[k]
        except Exception:
            pass

# ---- Buttons -------------------------------------------------------------
btn_load    = widgets.Button(description="Load/Reload overlay",
                             button_style="info")
btn_apply   = widgets.Button(description="Apply effects",
                             button_style="primary")
btn_off     = widgets.Button(description="All effects off")
btn_safe    = widgets.Button(description="Safe clean (mode 2)",
                             button_style="success")
btn_panic   = widgets.Button(description="Panic / mute (mode 3)",
                             button_style="danger")
btn_clear   = widgets.Button(description="Clear status counters")
btn_refresh = widgets.Button(description="Refresh status")

btn_mode_tone  = widgets.Button(description="Mode 0: tone")
btn_mode_dsp   = widgets.Button(description="Mode 2: DSP",
                                button_style="info")
btn_mode_mute  = widgets.Button(description="Mode 3: mute",
                                button_style="warning")

loop_confirm = widgets.Checkbox(value=False, description="confirm loopback")
btn_mode_loop  = widgets.Button(description="Mode 1: ADC->DAC loopback",
                                button_style="warning")

def on_load(_):
    load_overlay()
    if pmod_mmio is not None:
        # Force mode 2 at load so the notebook is consistent.
        write_mode(MODE["dsp"], "dsp")
    refresh_status()

btn_load.on_click(on_load)
btn_apply.on_click(lambda _: (apply_effects(), refresh_status()))
btn_off.on_click(lambda _: (all_effects_off(), refresh_status()))
btn_safe.on_click(lambda _: safe_clean())
btn_panic.on_click(lambda _: panic_mute())
btn_clear.on_click(lambda _: (clear_counters(), refresh_status()))
btn_refresh.on_click(lambda _: refresh_status())
btn_mode_tone.on_click(lambda _: (write_mode(MODE["tone"], "tone"), refresh_status()))
btn_mode_dsp.on_click( lambda _: (write_mode(MODE["dsp"],  "dsp"),  refresh_status()))
btn_mode_mute.on_click(lambda _: (write_mode(MODE["mute"], "mute"), refresh_status()))
def on_loopback(_):
    if not loop_confirm.value:
        _log("Mode 1 (loopback) requires the 'confirm loopback' checkbox. "
             "The DAC echoes the ADC sample at full scale and can feed "
             "back through the on-module Line Out <-> Line In jumper.")
        return
    write_mode(MODE["loopback"], "loopback")
    refresh_status()
btn_mode_loop.on_click(on_loopback)

# ---- Layout --------------------------------------------------------------
def _row(*ws):
    return widgets.HBox(list(ws))

ns_panel   = widgets.VBox([_row(ns_on,   ns_thr)])
comp_panel = widgets.VBox([_row(comp_on, comp_thr, comp_ratio),
                           _row(widgets.Label(""), comp_resp, comp_mk)])
od_panel   = widgets.VBox([_row(od_on, od_model),
                           _row(od_drive, od_tone, od_level)])
dist_panel = widgets.VBox([_row(dist_on, dist_pedal),
                           _row(dist_drive, dist_tone, dist_level)])
amp_panel  = widgets.VBox([_row(amp_on, amp_model),
                           _row(amp_in, amp_bass, amp_mid),
                           _row(amp_treb, amp_mst, amp_drv)])
cab_panel  = widgets.VBox([_row(cab_on, cab_model),
                           _row(cab_mix, cab_level)])
eq_panel   = widgets.VBox([_row(eq_on),
                           _row(eq_low, eq_mid, eq_high)])
rv_panel   = widgets.VBox([_row(rv_on),
                           _row(rv_decay, rv_tone, rv_mix)])

acc = widgets.Accordion(children=[
    ns_panel, comp_panel, od_panel, dist_panel,
    amp_panel, cab_panel, eq_panel, rv_panel,
])
acc.set_title(0, "Noise Suppressor")
acc.set_title(1, "Compressor")
acc.set_title(2, "Overdrive")
acc.set_title(3, "Distortion (pedal-mask)")
acc.set_title(4, "Amp Simulator")
acc.set_title(5, "Cab IR")
acc.set_title(6, "EQ")
acc.set_title(7, "Reverb")

title = widgets.HTML(
    "<h3>Pmod I2S2 Effect Control (mode 2 = ADC -> AudioLab DSP -> DAC, D49)</h3>"
    "<p style='color:#a00'><b>Disconnect the on-module Line Out &lt;-&gt; Line In jumper.</b> "
    "Put audio source on Line In at LOW volume. Listen on Line Out via a separate audio interface.</p>"
)

global_row1 = _row(btn_load, btn_apply, btn_off, btn_safe, btn_panic)
global_row2 = _row(btn_mode_tone, btn_mode_dsp, btn_mode_mute,
                   btn_mode_loop, loop_confirm)
global_row3 = _row(btn_clear, btn_refresh)

ui = widgets.VBox([
    title,
    global_row1,
    global_row2,
    global_row3,
    widgets.HTML("<b>pmod_status</b>"),
    status_out,
    widgets.HTML("<b>Effects</b>"),
    acc,
    widgets.HTML("<b>Log</b>"),
    log_out,
])

display(ui)

# Auto-load + mode 2 + initial status read so the cell is truly one-shot.
on_load(None)
apply_effects()
refresh_status()